# Week 8: Vector Workflows in Python

This notebook replicates QGIS vector operations using GeoPandas:
- Load and clean spatial data
- Perform spatial joins
- Calculate density metrics
- Create choropleth maps

---

## Data options

You have two choices:

| Option | Description |
|--------|-------------|
| **A. Sample data** | Use built-in sample data (NYC neighbourhoods + 311 complaints). No setup needed! |
| **B. Your own data** | Use data you exported from QGIS in earlier weeks |

**Recommendation:** Start with sample data to learn the workflow, then try your own data later.

---

## Step 0: Set up your environment

Run this cell first. It detects your environment and installs required packages.

In [ ]:
# Detect environment and install packages
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing GIS packages (takes ~1 minute)...")
    !pip install geopandas contextily mapclassify -q
    print("Done!")
else:
    print("Running in local Jupyter")
    print("Make sure you activated your conda environment: conda activate intro-gis")

---

## Step 1: Set up folders and paths

This creates the folder structure for saving your outputs.

In [ ]:
from pathlib import Path

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set paths to your data folders in Drive
    RAW = Path("/content/drive/MyDrive/intro-gis/week08/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week08/data/processed")
else:
    # Local paths (relative to notebook location)
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

# Create folders if they don't exist
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW}")
print(f"Processed folder: {PROCESSED}")

---

## Step 2: Import libraries

These are the Python tools we'll use.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

---

## Step 3: Load data

This cell checks if you have local data files. If not, it downloads sample data automatically.

### Sample data

The sample data uses **New York City**:
- **Neighbourhoods:** NYC Neighbourhood Tabulation Areas (NTAs) from NYC Open Data
- **Incidents:** 311 service requests (complaints/reports from residents)

### Your own data

To use your own data, place these files in `data/raw/`:
- `neighbourhoods.geojson` — polygon boundaries (e.g., SA2s from Week 3)
- `incidents.geojson` — point locations (e.g., crime data from Week 5)

**To export from QGIS:** Right-click layer → Export → Save Features As → Format: GeoJSON → CRS: EPSG:4326

In [ ]:
# Check if local files exist
local_neighbourhoods = RAW / "neighbourhoods.geojson"
local_incidents = RAW / "incidents.geojson"

if local_neighbourhoods.exists() and local_incidents.exists():
    # Use local files
    print("Found local data files - using your own data")
    neighbourhoods = gpd.read_file(local_neighbourhoods)
    incidents = gpd.read_file(local_incidents)
    USE_SAMPLE_DATA = False
else:
    # Download sample data
    print("Local files not found - downloading sample data (NYC)...")
    print("(To use your own data, add neighbourhoods.geojson and incidents.geojson to data/raw/)\n")
    
    # NYC Neighbourhood Tabulation Areas (NTAs)
    # Source: NYC Open Data
    nta_url = "https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd?method=export&format=GeoJSON"
    print("Downloading NYC neighbourhoods...")
    neighbourhoods = gpd.read_file(nta_url)
    
    # NYC 311 Service Requests (sample - recent complaints)
    # Source: NYC Open Data - limited to 5000 recent records for speed
    complaints_url = "https://data.cityofnewyork.us/resource/erm2-nwe9.geojson?$limit=5000&$where=latitude IS NOT NULL"
    print("Downloading NYC 311 complaints (sample of 5,000)...")
    incidents = gpd.read_file(complaints_url)
    
    USE_SAMPLE_DATA = True
    print("Done!\n")

print(f"Loaded {len(neighbourhoods)} neighbourhoods")
print(f"Loaded {len(incidents)} incidents")

# Preview the data
neighbourhoods.head(3)

---

## Step 4: Clean and prepare data

Standardize column names and identify the name field for joining.

In [ ]:
# Convert column names to lowercase (avoids case-sensitivity issues)
neighbourhoods.columns = neighbourhoods.columns.str.lower()
incidents.columns = incidents.columns.str.lower()

# Identify the name column (differs between sample and user data)
if USE_SAMPLE_DATA:
    # NYC data uses 'ntaname' for neighbourhood names
    NAME_COL = 'ntaname'
else:
    # Australian SA2 data uses 'sa2_name21'
    # Adjust this if your data uses a different column!
    NAME_COL = 'sa2_name21'
    if NAME_COL not in neighbourhoods.columns:
        print(f"Warning: '{NAME_COL}' not found. Available columns:")
        print(list(neighbourhoods.columns))
        print("\nSet NAME_COL to your neighbourhood name column, e.g.:")
        print("NAME_COL = 'suburb_name'")

# Calculate area in km² (need to project to meters first)
neighbourhoods["area_km2"] = neighbourhoods.to_crs(3857).area / 1e6

print(f"Using '{NAME_COL}' as the neighbourhood name column")
print(f"\nNeighbourhood columns: {list(neighbourhoods.columns)}")
neighbourhoods[[NAME_COL, "area_km2"]].head()

---

## Step 4b: Quick map to check the data

Always visualize your data before analysis to catch any issues.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot neighbourhoods
neighbourhoods.plot(ax=ax, color='lightgray', edgecolor='white', linewidth=0.5)

# Plot incidents on top
incidents.plot(ax=ax, color='red', markersize=1, alpha=0.3)

ax.set_title(f"Neighbourhoods ({len(neighbourhoods)}) and Incidents ({len(incidents)})")
ax.set_axis_off()
plt.show()

---

## Step 5: Spatial join

Count how many incidents fall within each neighbourhood. This is equivalent to:
- QGIS: `Vector > Data Management > Join Attributes by Location (Summary)`

In [ ]:
# Ensure same CRS
if neighbourhoods.crs != incidents.crs:
    incidents = incidents.to_crs(neighbourhoods.crs)
    print(f"Reprojected incidents to {neighbourhoods.crs}")

# Spatial join: link each incident to its neighbourhood
joined = gpd.sjoin(incidents, neighbourhoods, predicate="within", how="left")

# Count incidents per neighbourhood
counts = joined.groupby(NAME_COL).size().rename("incident_count")

# Merge counts back to neighbourhoods
neighbourhoods = neighbourhoods.merge(counts, on=NAME_COL, how="left")
neighbourhoods["incident_count"] = neighbourhoods["incident_count"].fillna(0)

print(f"Spatial join complete!")
print(f"\nTop 5 neighbourhoods by incident count:")
neighbourhoods.nlargest(5, "incident_count")[[NAME_COL, "incident_count"]]

---

## Step 6: Calculate incident rate

Incidents per square kilometre (normalizes for area size).

**Why normalize?** Larger neighbourhoods naturally have more incidents. Rate per km² lets us compare fairly.

In [ ]:
neighbourhoods["rate_per_km2"] = neighbourhoods["incident_count"] / neighbourhoods["area_km2"]

print("Top 5 neighbourhoods by incident RATE:")
neighbourhoods.nlargest(5, "rate_per_km2")[[NAME_COL, "incident_count", "area_km2", "rate_per_km2"]]

---

## Step 7: Create a choropleth map

Visualize the incident rate. This is equivalent to graduated symbology in QGIS.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

neighbourhoods.plot(
    column="rate_per_km2",
    scheme="quantiles",       # Classification method (like QGIS Mode)
    k=5,                       # Number of classes
    cmap="YlOrRd",            # Color ramp
    legend=True,
    legend_kwds={"title": "Rate per km²"},
    edgecolor="white",
    linewidth=0.3,
    ax=ax
)

ax.set_title("Incident Rate by Neighbourhood", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---

## Step 8: Export for QGIS

Save your results to `data/processed/` as a GeoPackage so you can open it in QGIS for final cartography.

In [ ]:
# Create output path
output_path = PROCESSED / "neighbourhoods_with_incidents.gpkg"

# Save to GeoPackage
neighbourhoods.to_file(output_path, driver="GPKG")

print(f"Saved to: {output_path}")
print("\nYou can now open this file in QGIS for professional cartography!")

---

## Done!

You've completed a full vector analysis workflow in Python:

1. Loaded spatial data (sample or your own)
2. Cleaned and prepared attributes
3. Performed a spatial join
4. Calculated density metrics
5. Created a choropleth visualization
6. Exported results for QGIS

---

## Try it yourself

Now that you've learned the workflow with sample data:

1. **Export your Week 3 boundaries** from QGIS as `neighbourhoods.geojson`
2. **Export your Week 5 crime points** as `incidents.geojson`
3. Place both in `data/raw/` and re-run the notebook

The notebook will automatically detect and use your files!

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`